# Gait-ViViT: A Video Processing Model for Parkinson's Disease Detection

In [ ]:
# Required libraries.
import os
import json
import cv2
import numpy as np
import subprocess
from pathlib import Path
from glob import glob
import torch
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import cv2
import numpy as np
import os
from pathlib import Path
try:
  import imageio
except ImportError:
  # Use imageio with ffmpeg for video format compatibility.
  !pip install imageio imageio-ffmpeg
  import imageio
import pandas as pd

## 1 - Dataset Setup

The datasets used for this project are [Connie et al.'s Kaggle MMU Visual-Based Parkinson's Disease Dataset](https://www.kaggle.com/datasets/teeconnie/mmu-visual-based-parkinsons-disease-dataset), which is hosted on Kaggle, and an internal dataset from Sapienza University of Rome.

### Kaggle MMU Visual-Based Parkinson's Disease Dataset

This dataset contains data extracted from **292 videos** of 167 subjects, of whom **93 are healthy** and **74 have Parkinson's disease**, walking.

The dataset is organized in **four folders**.
| Folder | Files | Description |
| :---: | :---: | :---: |
| `NORMAL` | 150 | Healthy subjects |
| `MILD` | 29 | Subjects with mild symptoms of Parkinson's disease |
| `MODERATE` | 61 | Subjects with moderate symptoms of Parkinson's disease |
| `SEVERE` | 62 | Subjects with severe symptoms of Parkinson's disease |

In [ ]:
!pip install --upgrade kaggle

In [ ]:
from google.colab import userdata

# Use the Kaggle Access Token that was saved using Google Colab Secrets.
os.environ["KAGGLE_TOKEN"] = userdata.get("KAGGLE_TOKEN")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Download and decompress the Kaggle MMU Dataset.
!mkdir -p /content/drive/MyDrive/bachelor_thesis/kaggle_json
!kaggle datasets download -d teeconnie/mmu-visual-based-parkinsons-disease-dataset -p /content/drive/MyDrive/bachelor_thesis/kaggle_json
!unzip -q -o /content/drive/MyDrive/bachelor_thesis/kaggle_json/mmu-visual-based-parkinsons-disease-dataset.zip -d /content/drive/MyDrive/bachelor_thesis/kaggle_json
!rm -f /content/drive/MyDrive/bachelor_thesis/kaggle_json/mmu-visual-based-parkinsons-disease-dataset.zip

print("Kaggle MMU Dataset successfully downloaded.")

Note that, instead the original videos, this dataset consists of just **JSON files** containing the **information extracted from each video**, likely due to privacy compliance reasons.

This information has been extracted using AlphaPose's point extraction method, which uses a heatmap for each keypoint and **extracts the keypoint with highest probability**.

For this reason, information is presented as a sequence of keypoints containing **spatial coordinates** and their **confidence**, which denotes the probability of the corresponding keypoint.

Therefore, the first step is to use the information contained in the JSON files to create **synthetic MP4 videos** containing the subject's **skeleton** during the video.

Each video contains the rendered skeleton, which is created using [AlphaPose's Halpe Full-Boy Human Keypoints](https://github.com/Fang-Haoshu/Halpe-FullBody), on a black background.

Given the structure of the JSON files in the dataset, this rendering function uses the `OpenCV` library to create the synthetic videos.

In [ ]:
# Halpe-Body Human Keypoints: https://github.com/Fang-Haoshu/Halpe-FullBody
HALPE_SKELETON = [
    (0, 1), (0, 2), (1, 3), (2, 4),               # Head and face
    (5, 18), (6, 18),                             # Shoulders and neck
    (5, 7), (7, 9),                               # Left arm
    (6, 8), (8, 10),                              # Right arm
    (18, 19),                                     # Neck and pelvis
    (11, 19), (12, 19),                           # Pelvis and hips
    (11, 13), (13, 15),                           # Left leg
    (12, 14), (14, 16),                           # Right leg
    (15, 24), (15, 20), (20, 22),                 # Left foot
    (16, 25), (16, 21), (21, 23)                  # Right foot
]

def json_to_video_kaggle(json_path, output_path, width=1280, height=720, fps=30):
  # Create the output directory.
  output_dir = os.path.dirname(output_path)
  if output_dir:
    os.makedirs(output_dir, exist_ok=True)

  # Load the JSON keypoint file.
  with open(json_path, "r") as f:
    frames_data = json.load(f)

  # Create a temporary file for saving the raw video.
  temp_output_path = output_path.replace(".mp4", "_temp.mp4")

  # Initialize OpenCV's VideoWriter to create .mp4 videos.
  fourcc = cv2.VideoWriter_fourcc(*"mp4v")
  out = cv2.VideoWriter(temp_output_path, fourcc, fps, (width, height))

  if not out.isOpened():
    raise RuntimeError(f"Error during initialization while processing {temp_output_path}.")

  for frame_info in frames_data:
    # Create a black background for each frame.
    img = np.zeros((height, width, 3), dtype=np.uint8)

    # Extract keypoints.
    people = frame_info.get("people", [frame_info])
    for person in people:
      keypoints = person.get("keypoints", [])
      points = []

      '''
      JSON data contain spatial coordinates and confidence scores.
      In fact, the model samples keypoints according to a heatmap, choosing the point with highest probability.
      The probability of the sampled point represents the model's confidence during extraction.
      '''

      for i in range(0, len(keypoints), 3):
        x, y, conf = keypoints[i], keypoints[i+1], keypoints[i+2]

        # Eliminate low-confidence points for denoising.
        if conf > 0.2:
          pt = (int(x), int(y))
          cv2.circle(img, pt, radius=4, color=(0, 255, 0), thickness=-1)
          points.append(pt)
        else:
          points.append(None)

        # Draw the bones according to the keypoint tuples.
        for connection in HALPE_SKELETON:
          idx1, idx2 = connection

          # Check that the two indices do not fall out of range.
          if idx1 < len(points) and idx2 < len(points):
            pt1 = points[idx1]
            pt2 = points[idx2]

            # Draw the bone if and only if both points have been detected.
            if pt1 is not None and pt2 is not None:
              cv2.line(img, pt1, pt2, color=(255, 0, 0), thickness=2) # Remember that OpenCV uses the BGR scheme.

    out.write(img)

  out.release()

  # Convert the raw video to the correct format.
  cmd = f"ffmpeg -y -i {temp_output_path} -c:v libx264 -pix_fmt yuv420p {output_path}"
  subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

  if os.path.exists(temp_output_path):
    os.remove(temp_output_path)

  print(f"{json_path} converted to synthetic video: {output_path}.")

In [ ]:
# Convert all JSON files in the Kaggle MMU Dataset into synthetic MP4 videos.
kaggle_json_dir = Path("/content/drive/MyDrive/bachelor_thesis/kaggle_json")
kaggle_mp4_dir = Path("/content/drive/MyDrive/bachelor_thesis/kaggle_videos")
kaggle_mp4_dir.mkdir(parents=True, exist_ok=True)

for el in kaggle_json_dir.rglob("*.json"):
  relative_json_path = el.relative_to(kaggle_json_dir)
  relative_mp4_path = relative_json_path.with_suffix(".mp4") # Change suffix from .json to .mp4.
  drive_mp4_path = kaggle_mp4_dir / relative_mp4_path
  drive_mp4_path.parent.mkdir(parents=True, exist_ok=True)

  # Since the procedure may interrupt, skip any file that has already been processed.
  if drive_mp4_path.exists():
    continue

  print(f"Converting {el} into a synthetic video.")
  json_to_video_kaggle(str(el), str(drive_mp4_path))

print("Kaggle MMU Dataset successfully rendered.")

### Internal GAIT Dataset

This dataset contains **2448 videos**, incuding depth, infrared, and RGB versions, of various **healthy subjects** walking or moving up/down the stairs.

Since the videos come in **AVI** format, the first step is to **extract spatial coordinates and confidence scores** using the AlphaPose Halpe model and using these keypoints to create **synthetic MP4 videos**.

Note that, since rendering the entire dataset can be problematic or unnecessary, only a portion of this dataset will actually be used.

Since the AlphaPose model has issues with processing videos in AVI format, the first step requires **converting the videos to MP4 format**.

In [ ]:
gait_avi = Path("/content/drive/MyDrive/bachelor_thesis/GAIT/dataset_blurred")
gait_mp4 = Path("/content/drive/MyDrive/bachelor_thesis/internal_mp4/dataset_blurred")
gait_json = Path("/content/drive/MyDrive/bachelor_thesis/internal_json/dataset_blurred")

gait_mp4.mkdir(parents=True, exist_ok=True)
gait_json.mkdir(parents=True, exist_ok=True)

# Start converting videos from .avi to .mp4 for compatibility with OpenCV modules used by AlphaPose.
avi_videos = sorted(list(gait_avi.rglob("*.avi")))
print(f"{len(avi_videos)} videos found to convert.")

for avi_path in avi_videos:
  # Create the target path for the current video.
  relative_path = avi_path.relative_to(gait_avi)
  mp4_path = gait_mp4 / relative_path.with_suffix(".mp4")
  mp4_path.parent.mkdir(parents=True, exist_ok=True)

  # Since the procedure may interrupt, skip any videos that have been previously converted.
  if not os.path.exists(mp4_path):
    print(f"Converting {relative_path}.")
    cmd = f"ffmpeg -y -i '{str(avi_path)}' -vcodec libx264 -pix_fmt yuv420p '{str(mp4_path)}' -loglevel error"
    subprocess.run(cmd, shell=True)

print("GAIT Dataset converted to .mp4 format.")

Since the rendering procedure tends to be computationally heavy, make sure to set *Runtime ▶ Change runtime type ▶ T4 GPU* before executing this section.

In [ ]:
print("A GPU is Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
# Install the AlphaPose Halpe model.
!pip install ninja yacs cython matplotlib tqdm opencv-python tensorboardX --quiet

%cd /content
if not os.path.exists("/content/AlphaPose"):
  !git clone https://github.com/MVIG-SJTU/AlphaPose.git

%cd /content/AlphaPose
!python setup.py build develop --quiet

# For simplicity, download the model only if it was not pre-downloaded.
!mkdir -p pretrained_models
drive_model_path = "/content/drive/MyDrive/bachelor_thesis/halpe26_fast_res50_256x192.pth"
if os.path.exists(drive_model_path):
  print("Copying the model from Google Drive.")
  !cp {drive_model_path} pretrained_models/
else:
  print("Downloading the model.")
  !gdown --id 1S-ROA28de-1zvLv-hVfPFJ5tFBYOSITb -O pretrained_models/halpe26_fast_res50_256x192.pth
  os.makedirs("/content/drive/MyDrive/bachelor_thesis", exist_ok=True)
  !cp pretrained_models/halpe26_fast_res50_256x192.pth {drive_model_path}

print("AlphaPose model ready for use.")

Remember to install the `cython_bbox` package and to fetch the `YOLO` weights used by the AlphaPose model.

In [ ]:
!pip install cython_bbox

In [ ]:
!mkdir -p /content/AlphaPose/detector/yolo/data
!wget -O /content/AlphaPose/detector/yolo/data/yolov3-spp.weights https://pjreddie.com/media/files/yolov3-spp.weights

In [ ]:
def process_single_video(mp4_path, is_first=False):
  # GPU-parallelized multithreaded video processing function.
  relative_mp4_path = mp4_path.relative_to(gait_mp4_root)
  target_json_file = gait_json_root / relative_mp4_path.parent / f"{mp4_path.stem}.json"

  # Since the procedure may interrupt, skip any video that has been previously processed.
  if target_json_file.exists():
    return

  # Use id(mp4_path) to avoid ambiguity when multiple threads are running in parallel.
  unique_prefix = id(mp4_path)
  local_video_path = local_temp_dir / f"{unique_prefix}_{mp4_path.name}"
  local_out_dir = local_temp_dir / f"{unique_prefix}_{mp4_path.stem}"
  local_out_dir.mkdir(parents=True, exist_ok=True)

  print(f"\n[START] Starting: {relative_mp4_path}", flush=True)

  try:
    # Work on a local copy.
    shutil.copy(str(mp4_path), str(local_video_path))
    cmd = [
          "python",
          "scripts/demo_inference.py",
          "--cfg", "configs/halpe_26/resnet/256x192_res50_lr1e-3_1x.yaml",
          "--checkpoint", checkpoint_path,
          "--video", str(local_video_path),
          "--outdir", str(local_out_dir),
          "--format", "cmu"
    ]

    # Since the first video needs to load the AlphaPose model, it gets a larger timeout.
    timeout = 360 if is_first else 180

    # Run the subprocess.
    subprocess.run(cmd, cwd="/content/AlphaPose", timeout=timeout, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

  except subprocess.TimeoutExpired:
    print(f"\n[TIMEOUT] Unlocking processes for {relative_mp4_path}", flush=True)

  except Exception as e:
    print(f"\n[ERROR] Unexpected error on {relative_mp4_path}: {e}", flush=True)

  extracted_file = local_out_dir / "alphapose-results.json"

  # Save the result on Google Drive and clean the local space on Google Colab.
  if extracted_file.exists():
    target_json_file.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(extracted_file), str(target_json_file))
    print(f"\n[SUCCESS] {relative_mp4_path}.", flush=True)
  else:
    print(f"\n[FAIL] {relative_mp4_path}.", flush=True)

  shutil.rmtree(local_out_dir, ignore_errors=True)
  if local_video_path.exists():
    local_video_path.unlink()

In [ ]:
gait_mp4_root = Path("/content/drive/MyDrive/bachelor_thesis/internal_mp4/dataset_blurred")
gait_json_root = Path("/content/drive/MyDrive/bachelor_thesis/internal_json/dataset_blurred")

mp4_videos = sorted(list(gait_mp4_root.rglob("*.mp4")))
print(f"Videos found to process: {len(mp4_videos)}")

checkpoint_path = "/content/drive/MyDrive/bachelor_thesis/halpe26_fast_res50_256x192.pth"

# Since the procedure may interrupt, skip any video that has been previously processed.
videos_to_process = []
for mp4_path in mp4_videos:
  relative_mp4_path = mp4_path.relative_to(gait_mp4_root)
  target_json_file = gait_json_root / relative_mp4_path.parent / f"{mp4_path.stem}.json"
  if not target_json_file.exists():
    videos_to_process.append(mp4_path)

print(f"{len(videos_to_process)} videos left to process.")

# Create a local folder to make the workflow less problematic.
local_temp_dir = Path("/content/temp_processing")
local_temp_dir.mkdir(parents=True, exist_ok=True)

if videos_to_process:
  # The first video is processed separately to avoid race conditions when loading the AlphaPose model.
  print("Processing the first video and initializing the AlphaPose model.")
  first_video = videos_to_process[0]
  process_single_video(first_video, is_first=True)

  # The remaining videos are processed in parallel.
  remaining_videos = videos_to_process[1:]

  # Set the number of threads based on the chosen GPU.
  # The L4 GPU (24 GB) can support 2-3 threads, whereas the A100 GPU (40 GB) can support 4-5 threads.
  MAX_WORKERS = 2

  if remaining_videos:
    print(f"Processing the remaining {len(remaining_videos)} using {MAX_WORKERS} threads.")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
      # Submit all tasks and gradually wait for completion.
      futures = [executor.submit(process_single_video, video, False) for video in remaining_videos]
      for future in as_completed(futures):
        try:
          future.result()
        except Exception as e:
          print(f"Error caused by thread: {e}", flush=True)

print("Keypoint extraction successfully completed.")

After extracting the keypoints from each video, it is possible to **render the JSON files in synthetic videos**, slightly adapting the method used for the Kaggle MMU Visual-Based Parkinson's Disease Dataset to make it compatible with how AlphaPose extracts and stores the keypoints.

Due to the format of the JSON files, this rendering function uses both the `OpenCV` and `Imageio` libraries for better flexibility.

In [ ]:
# Halpe-Body Human Keypoints: https://github.com/Fang-Haoshu/Halpe-FullBody
HALPE_SKELETON = [
    (0, 1), (0, 2), (1, 3), (2, 4),               # Head and face
    (5, 18), (6, 18),                             # Shoulders and neck
    (5, 7), (7, 9),                               # Left arm
    (6, 8), (8, 10),                              # Right arm
    (18, 19),                                     # Neck and pelvis
    (11, 19), (12, 19),                           # Pelvis and hips
    (11, 13), (13, 15),                           # Left leg
    (12, 14), (14, 16),                           # Right leg
    (15, 24), (15, 20), (20, 22),                 # Left foot
    (16, 25), (16, 21), (21, 23)                  # Right foot
]

# Adapt the width and height parameters to the original video resolution.
def json_to_video_v2(json_path, output_path, width=848, height=480, fps=30):
  # Create the output directory.
  output_dir = os.path.dirname(output_path)
  if output_dir:
    os.makedirs(output_dir, exist_ok=True)

  # Load JSON data.
  with open(json_path, 'r') as f:
    frames_dict = json.load(f)

  # Since JSON extraction is done is parallel, sort the frames first.
  sorted_frame_keys = sorted(frames_dict.keys(), key=lambda x: int(x.split('.')[0]))

  # Initialize Imageio's writer to create compatible videos.
  writer = imageio.get_writer(output_path, fps=fps, codec='libx264', pixelformat='yuv420p')

  # Iterate through frames.
  for frame_key in sorted_frame_keys:
    frame_info = frames_dict[frame_key]

    img = np.zeros((height, width, 3), dtype=np.uint8)

    # Extract data from the bodies/joints structure.
    bodies = frame_info.get('bodies', [])

    for body in bodies:
      keypoints = body.get('joints', [])
      points = []

      for i in range(0, len(keypoints), 3):
        x, y, conf = keypoints[i], keypoints[i+1], keypoints[i+2]

        # Eliminate low-confidence points.
        if conf > 0.2:
          pt = (int(x), int(y))
          cv2.circle(img, pt, radius=4, color=(0, 255, 0), thickness=-1)
          points.append(pt)
        else:
          points.append(None)

      # Draw the bones using lines joining point tuples from the Keypoints.
      for connection in HALPE_SKELETON:
        idx1, idx2 = connection
        # Check that both indices do not fall out of range.
        if idx1 < len(points) and idx2 < len(points):
          pt1 = points[idx1]
          pt2 = points[idx2]
          # Draw the line if and only if both points got detected.
          if pt1 is not None and pt2 is not None:
            cv2.line(img, pt1, pt2, color=(255, 0, 0), thickness=2)

    # Since OpenCV uses BGR but Imageio uses RGB, convert the colour channels.
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    writer.append_data(img_rgb)

  writer.close()
  print(f"{json_path} converted to a synthetic video: {output_path}")

In [ ]:
# Convert all JSON files in the Internal Dataset into synthetic MP4 videos..
gait_json_dir = Path("/content/drive/MyDrive/bachelor_thesis/internal_json")
gait_mp4_dir = Path("/content/drive/MyDrive/bachelor_thesis/internal_videos") # Save on Google Drive for future mounting.
gait_mp4_dir.mkdir(parents=True, exist_ok=True)

print(f"Videos found to process: {len(list(gait_json_dir.rglob("*.json")))}")

# Iterate through the JSON files.
for el in gait_json_dir.rglob("*.json"):
  relative_json_path = el.relative_to(gait_json_dir)
  relative_mp4_path = relative_json_path.with_suffix(".mp4") # Change suffix from .json to .mp4.
  drive_mp4_path = gait_mp4_dir / relative_mp4_path
  drive_mp4_path.parent.mkdir(parents=True, exist_ok=True)

  # If the JSON file has already been converted to a synthetic video, skip it.
  if drive_mp4_path.exists():
    continue

  print(f"Converting {el} into a synthetic video")
  json_to_video_v2(str(el), str(drive_mp4_path)) # Convert Path objects to strings.

print("Internal Dataset successfully rendered.")

## Dataframe Preparation

After downloading the two datasets, the next step consists of creating a **unified dataframe** that contains information about each video.

This dataframe will contain three fields:
1. `video_id`, which provides a unique identifier for each video.
2. `source`, which indicates whether the video comes from the Kaggle MMU Dataset or from the internal dataset.
3. `path`, which represents the path to the rendered video.
4. `width`, which denotes the width of each video frame.
5. `height`, which denotes the height of each video frame.
6. `parkinson`, which indicates whether the subject is healthy (`parkinson = 0`) or has Parkinson's disease (`parkinson = 1`).

In [ ]:
rows = []

# Add videos from the Kaggle MMU Dataset.
# Determine the labels for each class in the dataset.
kaggle_labels = {
    "normal": 0,
    "mild": 1,
    "moderate": 1,
    "severe": 1
}

kaggle_base_dir = "/content/drive/MyDrive/bachelor_thesis/kaggle_videos"

# Use os.walk to automate file search.
for root, dirs, files in os.walk(kaggle_base_dir):
  for file in files:
    if file.endswith(".mp4"):
      mp4_path = os.path.join(root, file)
      folder_name = os.path.basename(root).lower()

      if folder_name in kaggle_labels:
        video_id = os.path.splitext(file)[0]
        val = kaggle_labels[folder_name]

        rows.append({
            "video_id": video_id,
            "source": "Kaggle",
            "path": mp4_path,
            "width": 1280,
            "height": 720,
            "parkinson": val
        })

print("Kaggle MMU Dataset successfully extracted.")

# Add videos from the GAIT Dataset.
gait_base_dir = "/content/drive/MyDrive/bachelor_thesis/internal_videos"

# Use os.walk to automate file search.
for root, dirs, files in os.walk(gait_base_dir):
  for file in files:
    if file.endswith(".mp4"):
      video_path = os.path.join(root, file)
      # Use the video's relative path to create a unique ID.
      relative_path = os.path.relpath(video_path, gait_base_dir)
      video_id = os.path.splitext(relative_path)[0].replace(os.sep, "-")

      # Skip depth and ir videos to avoid dirty data.
      if "rgb" not in video_path:
        continue

      # Since all subjects in the GAIT Dataset are healthy, set parkinson = 0 for each.
      rows.append({
          "video_id": video_id,
          "source": "Internal",
          "path": video_path,
          "width": 848,
          "height": 480,
          "parkinson": 0
      })

print("Internal Dataset successfully extracted.")

# Aggregation and merging.
unified_df = pd.DataFrame(rows)

print(f"Total videos: {len(unified_df)}\n")

print("Count based on the source dataset:")
print(unified_df['source'].value_counts())
print("\nClass balance (0 = Healthy, 1 = Parkinson):")
print(unified_df['parkinson'].value_counts())

# Save the dataframe.
df_dir = "/content/drive/MyDrive/bachelor_thesis/dataframes"
os.makedirs(df_dir, exist_ok=True)
unified_df.to_csv("/content/drive/MyDrive/bachelor_thesis/dataframes/unified_dataset.csv", index=False)